### Kelompok: Farant Marchelino | 2024081012, Natasya Olivia Safitri | 2025011020

### 1. IMPORT LIBRARY
Memuat semua pustaka dasar untuk pengolahan data dan *Machine Learning*.

In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### 2. DATASET
Memuat data dari file CSV hasil survei perilaku konsumen.

In [20]:
file_path = 'Dataset Rekomendasi Produk.csv'
df = pd.read_csv(file_path)
df.head()

,Cap waktu,Nama Lengkap,Jenis Kelamin,Usia Anda saat ini,Pekerjaan Utama,Rata-rata frekuensi belanja online dalam satu bulan,Platform e-commerce yang paling sering Anda gunakan: (Pilih satu),"1. Seberapa sering Anda melihat/memperhatikan kolom ""Rekomendasi Untuk Anda"" saat membuka aplikasi belanja online?",2. Jenis rekomendasi mana yang paling sering membuat Anda tertarik untuk mengklik produk tersebut? (Pilih satu yang paling dominan),3. Seberapa akurat menurut Anda sistem rekomendasi aplikasi dalam menebak barang yang memang sedang Anda butuhkan?,...,6. Apa alasan utama Anda mengabaikan/melewati produk yang direkomendasikan?,"1. Sejauh mana faktor-faktor berikut memengaruhi Anda untuk mengeklik/membeli produk dari hasil rekomendasi AI?\n \n(1 = Sangat Tidak Berpengaruh, 5 = Sangat Berpengaruh) [Rating & Ulasan]","1. Sejauh mana faktor-faktor berikut memengaruhi Anda untuk mengeklik/membeli produk dari hasil rekomendasi AI?\n \n(1 = Sangat Tidak Berpengaruh, 5 = Sangat Berpengaruh) [Visual/Foto Produk]","1. Sejauh mana faktor-faktor berikut memengaruhi Anda untuk mengeklik/membeli produk dari hasil rekomendasi AI?\n \n(1 = Sangat Tidak Berpengaruh, 5 = Sangat Berpengaruh) [Harga & Promo]","1. Sejauh mana faktor-faktor berikut memengaruhi Anda untuk mengeklik/membeli produk dari hasil rekomendasi AI?\n \n(1 = Sangat Tidak Berpengaruh, 5 = Sangat Berpengaruh) [Popularitas Toko (Mall/Star+)]",2. Saya merasa lebih nyaman berbelanja jika aplikasi memberikan rekomendasi produk yang sesuai dengan kepribadian saya.,3. Saya merasa sistem rekomendasi AI di e-commerce sangat membantu saya menghemat waktu dalam mencari barang.,"4. Jika sistem rekomendasi terus memunculkan produk yang tidak relevan, saya cenderung akan menutup aplikasi tersebut.",5. Manakah yang lebih Anda percayai saat memilih produk?,6. Apa ada saran atau harapan Anda agar sistem rekomendasi produk di aplikasi belanja online menjadi lebih baik dan berguna bagi Anda?
0,2026/04/10 12:56:31 PM GMT+7,Zidane Tirta Nugraha,Laki-laki,18 - 24 Tahun,Pelajar / Mahasiswa,Kurang dari 2 kali,Shopee,4,Rekomendasi berdasarkan barang yang baru saja ...,3,...,Produk tidak sesuai dengan selera/minat saya.;...,4,4,4,4,4,5,Ya,Hasil pencarian manual saya sendiri,Tidak Ada
1,2026/04/11 1:44:21 PM GMT+7,Fellicia Fernanda,Perempuan,< 18 Tahun,Pelajar / Mahasiswa,Kurang dari 2 kali,Shopee,5,Rekomendasi berdasarkan barang yang baru saja ...,5,...,Saya sudah membeli produk serupa sebelumnya.;H...,5,5,5,5,5,5,Ya,Rekomendasi dari teman/influencer di media sosial,Tidak Ada
2,2026/04/11 1:45:44 PM GMT+7,Muhammad Rayhan Mumtaz,Laki-laki,18 - 24 Tahun,Pelajar / Mahasiswa,3 - 5 kali,Shopee,4,Rekomendasi berdasarkan barang yang baru saja ...,4,...,Produk tidak sesuai dengan selera/minat saya.;...,4,4,5,5,4,5,Tidak,Hasil pencarian manual saya sendiri,Tidak Ada
3,2026/04/11 1:52:10 PM GMT+7,Mochammad Lintar Arya Dwiputra,Laki-laki,18 - 24 Tahun,Pelajar / Mahasiswa,Kurang dari 2 kali,Shopee,4,Rekomendasi berdasarkan barang yang baru saja ...,3,...,Saya sudah membeli produk serupa sebelumnya.;H...,4,4,5,5,5,5,Ya,Hasil pencarian manual saya sendiri,Saya berharap sistem rekomendasi produk bisa l...
4,2026/04/11 1:52:28 PM GMT+7,Fizar Erlansyah,Laki-laki,18 - 24 Tahun,Pelajar / Mahasiswa,3 - 5 kali,Shopee,4,Rekomendasi berdasarkan barang yang baru saja ...,4,...,Produk tidak sesuai dengan selera/minat saya.;...,5,5,5,5,5,4,Ya,Hasil pencarian manual saya sendiri,Tidak Ada


### 3. DATA PREPARATION & CLEANING
Menyeragamkan nama kolom dan membuang fitur yang tidak memiliki nilai prediktif.

In [21]:
df.columns = [
    'Cap_Waktu', 'Nama', 'Gender', 'Usia', 'Pekerjaan',
    'Frekuensi_Belanja', 'Platform',
    'Frekuensi_Lihat_Rekomendasi',
    'Jenis_Rekomendasi',
    'Akurasi_Rekomendasi',
    'Impulse_Buying',
    'Fitur_Meyakinkan',
    'Alasan_Abaikan',
    'Faktor_Rating',
    'Faktor_Visual',
    'Faktor_Harga',
    'Faktor_Popularitas',
    'Nyaman_Rekomendasi',
    'Hemat_Waktu',
    'Tutup_Jika_Tidak_Relevan',
    'Kepercayaan_Sumber',
    'Saran'
]

# Membersihkan kolom multi-pilihan (mengambil alasan pertama saja)
df['Alasan_Abaikan'] = df['Alasan_Abaikan'].astype(str).apply(lambda x: x.split(';')[0].strip())

# Menghapus kolom yang tidak relevan untuk Machine Learning
df_ml = df.drop(columns=['Cap_Waktu', 'Nama', 'Saran'])

### 4. ENCODING (TRANSFORMASI TEKS KE ANGKA)
Mengubah data kategorikal menjadi format numerik agar dapat diproses oleh algoritma.

In [22]:
le_dict = {}
kolom_kategorikal = [
    'Gender', 'Usia', 'Pekerjaan', 'Frekuensi_Belanja', 'Platform',
    'Jenis_Rekomendasi', 'Fitur_Meyakinkan', 'Alasan_Abaikan',
    'Tutup_Jika_Tidak_Relevan', 'Kepercayaan_Sumber'
]

for col in kolom_kategorikal:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))
    le_dict[col] = le

# Encoding Target/Label (Impulse_Buying: Tidak / Ya)
le_target = LabelEncoder()
df_ml['Impulse_Buying'] = le_target.fit_transform(df_ml['Impulse_Buying'].astype(str))

# Pemisahan Fitur (X) dan Target (y)
X = df_ml.drop(columns=['Impulse_Buying'])
y = df_ml['Impulse_Buying']

### 5. PEMBAGIAN DATA & SCALING
Membagi dataset menjadi 80% Latih dan 20% Uji. Fitur juga distandardisasi agar skalanya seimbang.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 6. PEMODELAN: NAIVE BAYES

In [24]:
nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
nb_pred = nb_model.predict(X_test_scaled)
nb_accuracy = accuracy_score(y_test, nb_pred)

### 7. PEMODELAN: LOGISTIC REGRESSION

In [25]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)

### 8. EVALUASI METRIK KINERJA MODEL
Mencetak *Classification Report* untuk membandingkan Presisi, Recall, dan F1-Score.

In [26]:
print("     PERBANDINGAN EFEKTIVITAS: NAIVE BAYES vs LOGISTIC REGRESSION  ")
print(f"Total Dataset          : {len(df)} Responden")
print(f"Jumlah Data Latih (80%): {len(X_train)} data")
print(f"Jumlah Data Uji (20%)  : {len(X_test)} data")
print("-------------------------------------------------------------------")
print(f"[1] AKURASI NAIVE BAYES        : {nb_accuracy * 100:.2f}%")
print(f"[2] AKURASI LOGISTIC REGRESSION: {lr_accuracy * 100:.2f}%")
print("-------------------------------------------------------------------")

# Menentukan model mana yang menang secara otomatis
if lr_accuracy > nb_accuracy:
    pemenang = "LOGISTIC REGRESSION"
    selisih = (lr_accuracy - nb_accuracy) * 100
elif nb_accuracy > lr_accuracy:
    pemenang = "NAIVE BAYES"
    selisih = (nb_accuracy - lr_accuracy) * 100
else:
    pemenang = "KEDUA MODEL SEIMBANG"
    selisih = 0.0

if selisih > 0:
    print(f"KESIMPULAN: Algoritma {pemenang} lebih efektif pada dataset ini")
    print(f"dengan selisih akurasi sebesar {selisih:.2f}%.")
else:
    print(f"KESIMPULAN: {pemenang} dalam efektivitas pada dataset ini.")

print("\n[Detail Laporan Classification - Naive Bayes]")
print(classification_report(y_test, nb_pred, target_names=le_target.classes_, zero_division=0))

print("[Detail Laporan Classification - Logistic Regression]")
print(classification_report(y_test, lr_pred, target_names=le_target.classes_, zero_division=0))

     PERBANDINGAN EFEKTIVITAS: NAIVE BAYES vs LOGISTIC REGRESSION  
Total Dataset          : 20 Responden
Jumlah Data Latih (80%): 16 data
Jumlah Data Uji (20%)  : 4 data
-------------------------------------------------------------------
[1] AKURASI NAIVE BAYES        : 75.00%
[2] AKURASI LOGISTIC REGRESSION: 75.00%
-------------------------------------------------------------------
KESIMPULAN: KEDUA MODEL SEIMBANG dalam efektivitas pada dataset ini.

[Detail Laporan Classification - Naive Bayes]
                     precision    recall  f1-score   support

              Tidak       0.75      1.00      0.86         3
Ya (Impulse Buying)       0.00      0.00      0.00         1

           accuracy                           0.75         4
          macro avg       0.38      0.50      0.43         4
       weighted avg       0.56      0.75      0.64         4

[Detail Laporan Classification - Logistic Regression]
                     precision    recall  f1-score   support

            

## Kesimpulan Analisis: Naive Bayes vs Logistic Regression
Dari hasil pemodelan yang dilakukan pada dataset survei perilaku konsumen, ditemukan bahwa **kedua algoritma (Naive Bayes dan Logistic Regression) memiliki tingkat akurasi yang identik, yaitu 75,00%.**

Performa Model: Kedua model mampu memprediksi kelas "Tidak" (konsumen yang tidak melakukan impulse buying) dengan presisi 75% dan recall 100%. Namun, kedua model gagal memprediksi kelas "Ya" (konsumen yang melakukan impulse buying), yang ditunjukkan dengan nilai precision dan recall sebesar 0,00 untuk kategori tersebut.

**Mengapa Hasilnya Sama?**
Hasil yang identik ini terjadi karena jumlah data yang digunakan sangat terbatas (hanya 20 responden). Dalam Machine Learning, algoritma memerlukan jumlah sampel yang cukup besar untuk dapat mengenali pola yang kompleks dan membedakan karakteristik antar kategori secara akurat. Dengan hanya 16 data latih dan 4 data uji, model belum mendapatkan "variasi" informasi yang cukup untuk menunjukkan perbedaan performa yang signifikan antara pendekatan probabilitas (Naive Bayes) dan pendekatan linear (Logistic Regression). Akibatnya, model cenderung memberikan prediksi yang sangat sederhana atau dominan pada kelas mayoritas saja.